# ДЗ 4. Операции, axis и broadcasting на данных «Дала Маркет»
Срок: **до понедельника 16.11, 19:00**.
Сдача: папка `hw04` репозитория `dsa-homework` (`hw04.ipynb`): `git add hw04` → `git commit` → `git push`, ссылку прикрепите в LMS.

Каждый ответ: **число + одна фраза для менеджера**. Циклы `for` по заказам не используем. Небольшой цикл по городам или категориям при сборке таблицы допустим.

- **[База]** обязательно для всех · **[Стандарт]** сколько получится · **[Челлендж]** для тех, кому легко.

## [Для всех] Чтение кода: без запуска
Сначала напишите ответ и «почему», потом запустите.
```python
import numpy as np
# 1
a = np.array([2, 4, 6]); print(a / 2, a // 4)
# 2
m = np.ones((3, 4)); print(m.sum(axis=1))
# 3
print(np.arange(3) + np.arange(3).reshape(3, 1))
# 4
x = np.array([1, 4, 9]); print(np.sqrt(x).sum())
# 5
print((np.arange(1, 5) ** 2).mean())
```

Ваши ответы и объяснения:

1. [1. 2. 3.] [0 1 1] — / делит поэлементно, а // даёт целую часть деления.
2. [4. 4. 4.] — в каждой из трёх строк четыре единицы, axis=1 суммирует по строкам.
3. [[0 1 2], [1 2 3], [2 3 4]] — broadcasting прибавляет строку (3,) к каждому из трёх элементов столбца (3,1).
4.6.0 — корни из 1, 4, 9: 1 + 2 + 3.
5. 7.5 — среднее квадратов 1, 4, 9, 16: 30 / 4.

In [ ]:
# Загрузка orders.csv: в Colab нажмите «Выбрать файлы» и укажите orders.csv из LMS.
# Если файл уже лежит рядом с ноутбуком, загрузка пропускается.
import os
if not os.path.exists("orders.csv"):
    try:
        from google.colab import files
        files.upload()
    except ImportError:
        print("Положите orders.csv в одну папку с ноутбуком")
print("orders.csv на месте:", os.path.exists("orders.csv"))

In [ ]:
import csv
import numpy as np

CITY = {"алматы": "Алматы", "almaty": "Алматы", "астана": "Астана", "astana": "Астана",
        "шымкент": "Шымкент", "shymkent": "Шымкент", "караганда": "Караганда",
        "актобе": "Актобе", "атырау": "Атырау"}

with open("orders.csv", encoding="utf-8") as f:
    rows = list(csv.DictReader(f))

amount = np.array([float(r["amount"].replace(" ", "")) for r in rows])
paid   = np.array([r["status"] == "оплачен" for r in rows])
days   = np.array([float(r["delivery_days"]) if r["delivery_days"] else np.nan for r in rows])
city   = np.array([CITY.get(r["city"].strip().lower(), r["city"].strip()) for r in rows])
cust   = np.array([r["customer_id"] for r in rows])
month  = np.array([r["order_date"][:7] for r in rows])      # "2026-09" или "2026-10"
cat    = np.array([r["category"] for r in rows])
oid    = np.array([r["order_id"] for r in rows])

print(amount.shape, np.unique(month))

In [ ]:
# Чистые данные: без дублей (оставляем первое вхождение order_id), без TEST, только оплаченные
_, first = np.unique(oid, return_index=True)
keep = np.zeros(len(oid), dtype=bool)
keep[first] = True
clean = keep & (cust != "TEST") & paid
print("строк всего:", len(oid), "| в чистых данных:", clean.sum())

## [База] 1. Выручка по чистым данным
Сколько строк осталось в `clean`? Выручка за сентябрь и октябрь и изменение в %. Совпадает ли с ДЗ 2?

In [8]:
# 1. Выручка по чистым данным (маска clean уже построена выше)
months = np.array(["2026-09", "2026-10"])
sep = amount[clean & (month == months[0])].sum()
oct_ = amount[clean & (month == months[1])].sum()
change_pct = (oct_ / sep - 1) * 100
print('Чистых оплаченных заказов:', clean.sum())
print(f'Сентябрь: {sep:,.0f} ₸')
print(f'Октябрь: {oct_:,.0f} ₸')
print(f'Изменение: {change_pct:+.2f}%')
print(f'Вывод менеджеру: выручка изменилась на {change_pct:+.2f}%; данные очищены от TEST и повторных заказов.')


Чистых оплаченных заказов: 4739
Сентябрь: 35,243,410 ₸
Октябрь: 34,419,520 ₸
Изменение: -2.34%
Вывод менеджеру: выручка изменилась на -2.34%; данные очищены от TEST и повторных заказов.


## [База] 2. Таблица «категория × месяц»
Соберите таблицу 5 × 2 по образцу таблицы городов с урока (категории: `np.unique(cat)`).
Какая категория приносит больше всего выручки за два месяца (`axis`)? Какая упала сильнее всех в %?

In [9]:
# 2
cats = np.unique(cat)

category_table = np.array([
    [
        amount[clean & (cat == c) & (month == m)].sum()
        for m in months
    ]
    for c in cats
])

print(category_table.shape)

for c, r in zip(cats, category_table):
    print(c, r)

totals = category_table.sum(axis=1)
changes = (category_table[:, 1] / category_table[:, 0] - 1) * 100

print("Лидер:", cats[totals.argmax()])
print("Сильнее упала:", cats[changes.argmin()])

print(f"Вывод менеджеру: по общей выручке лидирует {cats[totals.argmax()]}; сильнее всего снизилась категория {cats[changes.argmin()] }.")


(5, 2)
Дом [4997730. 5125790.]
Книги [1065300. 1140910.]
Красота [2610950. 2678990.]
Одежда [9167100. 8505730.]
Электроника [17402330. 16968100.]
Лидер: Электроника
Сильнее упала: Одежда
Вывод менеджеру: по общей выручке лидирует Электроника; сильнее всего снизилась категория Одежда.


## [База] 3. Средний чек по категориям в октябре
Соберите вторую таблицу: **число** заказов по категориям и месяцам (сумма маски, а не сумма amount).
Средний чек = выручка / число заказов, одной операцией над двумя таблицами. У какой категории он самый высокий?

In [10]:
# 3
count_table = np.array([
    [
        (clean & (cat == c) & (month == m)).sum()
        for m in months
    ]
    for c in cats
])

avg_check = category_table / count_table

print("Количество заказов:")
print(count_table)

print("Средний чек:")
print(np.round(avg_check, 2))

overall_avg = category_table.sum(axis=1) / count_table.sum(axis=1)

print("Лидер среднего чека:", cats[overall_avg.argmax()])

print(f"Вывод менеджеру: самый высокий средний чек за два месяца у категории {cats[overall_avg.argmax()]}.")


Количество заказов:
[[476 476]
 [265 299]
 [422 435]
 [678 652]
 [531 505]]
Средний чек:
[[10499.43 10768.47]
 [ 4020.    3815.75]
 [ 6187.09  6158.6 ]
 [13520.8  13045.6 ]
 [32772.75 33600.2 ]]
Лидер среднего чека: Электроника
Вывод менеджеру: самый высокий средний чек за два месяца у категории Электроника.


## [Стандарт] 4. Доля городов в выручке месяца
Используйте `table` городов с урока. Доля каждого города в выручке своего месяца: `table / table.sum(axis=0)`.
Проверьте, что каждый столбец в сумме даёт 1. Как изменилась доля Шымкента?

In [11]:
# 4. Доли городов в выручке каждого месяца
cities = np.unique(city)
table = np.array([
    [amount[clean & (city == c) & (month == m)].sum() for m in months]
    for c in cities
])
shares = table / table.sum(axis=0)
for c, share in zip(cities, shares):
    print(f'{c}: сентябрь {share[0] * 100:.2f}%, октябрь {share[1] * 100:.2f}%')
print('Проверка суммы долей:', shares.sum(axis=0))
# Маска выбирает строку Шымкента
shymkent = shares[cities == 'Шымкент'][0]
shymkent_change = (shymkent[1] - shymkent[0]) * 100
print(f'Шымкент: {shymkent[0] * 100:.2f}% → {shymkent[1] * 100:.2f}%, изменение {shymkent_change:+.2f} п.п.')
print(f'Вывод менеджеру: доля Шымкента изменилась на {shymkent_change:+.2f} п.п.; проверьте причины изменения доли.')


Актобе: сентябрь 7.31%, октябрь 7.38%
Алматы: сентябрь 41.15%, октябрь 42.54%
Астана: сентябрь 25.63%, октябрь 25.71%
Атырау: сентябрь 2.62%, октябрь 2.82%
Караганда: сентябрь 10.35%, октябрь 9.80%
Шымкент: сентябрь 12.93%, октябрь 11.75%
Проверка суммы долей: [1. 1.]
Шымкент: 12.93% → 11.75%, изменение -1.19 п.п.
Вывод менеджеру: доля Шымкента изменилась на -1.19 п.п.; проверьте причины изменения доли.


## [Стандарт] 5. Кто дал падение
Вклад каждого города в общее падение выручки: изменение города / общее изменение. Сколько процентов падения даёт Шымкент?
Напишите менеджеру одну фразу: где искать причину.

In [12]:
# 5. Вклад каждого города в общее изменение выручки
city_change = table[:, 1] - table[:, 0]
total_change = city_change.sum()
contribution = city_change / total_change * 100
for c, value in zip(cities, contribution):
    print(f'{c}: {value:+.2f}%')
shymkent_contrib = contribution[cities == 'Шымкент'][0]
print(f'Вывод менеджеру: Шымкент объясняет {shymkent_contrib:.2f}% чистого падения; положительные значения означают вклад в снижение, отрицательные — компенсацию роста.')


Актобе: +4.44%
Алматы: -17.23%
Астана: +22.30%
Атырау: -5.57%
Караганда: +33.60%
Шымкент: +62.46%
Вывод менеджеру: Шымкент объясняет 62.46% чистого падения; положительные значения означают вклад в снижение, отрицательные — компенсацию роста.


## [Стандарт] 6. Исправьте код от AI
AI написал код «доля сентября и октября внутри каждого города». Код падает. Объясните, почему, и исправьте.
```python
share = table / table.sum(axis=1)
```

In [13]:
# 6# 6. Исправляем broadcasting: keepdims=True сохраняет форму (6, 1)
# Ошибка AI: table имеет форму (6, 2), а суммы по строкам без keepdims — (6,).
# Формы (6,2) и (6,) несовместимы при делении справа налево.
share = table / table.sum(axis=1, keepdims=True)
for c, value in zip(cities, share):
    print(f'{c}: сентябрь {value[0] * 100:.2f}%, октябрь {value[1] * 100:.2f}%')
print('Проверка суммы долей внутри города:', share.sum(axis=1))
print('Вывод менеджеру: доли показывают распределение выручки каждого города между двумя месяцами.')


Актобе: сентябрь 50.36%, октябрь 49.64%
Алматы: сентябрь 49.76%, октябрь 50.24%
Астана: сентябрь 50.51%, октябрь 49.49%
Атырау: сентябрь 48.79%, октябрь 51.21%
Караганда: сентябрь 51.97%, октябрь 48.03%
Шымкент: сентябрь 52.99%, октябрь 47.01%
Проверка суммы долей внутри города: [1. 1. 1. 1. 1. 1.]
Вывод менеджеру: доли показывают распределение выручки каждого города между двумя месяцами.


## [Челлендж] 7. Стандартизация суммы заказа
Для чистых оплаченных заказов посчитайте z-оценку: `z = (x - x.mean()) / x.std()`.
Сколько заказов с |z| > 3 и какая это доля? Проверьте, что у `z` среднее ≈ 0 и std = 1.

In [ ]:
# 7

## [Челлендж] 8. Масштабирование признаков (заготовка для ML)
Соберите матрицу `X` из двух столбцов: `amount` и `days` для чистых заказов без пропусков в `days` (`np.column_stack`).
Приведите каждый столбец к диапазону [0, 1] одной строкой: `(X - X.min(axis=0)) / (X.max(axis=0) - X.min(axis=0))`.
Проверьте min и max каждого столбца. Объясните своими словами, зачем модели нужны признаки в одном масштабе.

In [ ]:
# 8